# Publish a Foundry agent to Teams via the SHARED container (notebook edition)

This notebook does exactly what the Streamlit publisher does, but step-by-step in code, so you can see how each agent is wired to the **shared** platform. It **reuses the tested `publisher.py` functions** — no per-agent container is created; each agent becomes a **registry row** that the one shared translator container reads.

**Shared platform this hooks into (deployed once — see `../setup/SETUP_SHARED_PLATFORM.md`):**
- **Shared OAuth client (Entra app)** — your pre-authorized client on the MCP resource app (no admin consent needed).
- **Shared translator Container App** — your registry-driven router container (`AGENT_REGISTRY_TABLE=agents`).
- **APIM** — your APIM (public front, `validate-jwt`, templated route `/agents/{agentId}/messages` + `/api/oauth/*`).
- **MCP server + resource app** — your MCP behind APIM (only needed for agents whose MCP tool requires per-user auth).

**Per agent (what this notebook does):** bot Entra app + secret → Key Vault, merge bot audience into APIM, derive auth mode from the agent's tools, create the Azure Bot + Teams channel, **write the registry row**, build the Teams `.zip`.

## Prerequisites

1. **`az login`** into your tenant (`az login --tenant <your-tenant-id>`).
2. **Network reach**: if Key Vault + the Storage table are private-endpoint only, connect your VPN / run from a machine inside (or peered to) the VNet.
3. **RBAC**: your account needs `Key Vault Secrets Officer` (bot secret) and `Storage Table Data Contributor` (registry row).
4. **`.env` configured** in `streamlit_app/` (shared IDs + `SHARED_OAUTH_CLIENT_ID/SECRET`, `DEFAULT_MCP_SCOPE`, `OAUTH_REDIRECT_BASE_URL`). This notebook reads that same `.env`.
5. Run this notebook with a Python environment that has the publisher's `requirements.txt` installed.

## 1. Point at the publisher code + load shared config

We add `streamlit_app/` to the path and import the same functions the Streamlit app uses. `get_config()` loads the shared platform IDs from `streamlit_app/.env`.

In [ ]:
import os, sys, pathlib

# This notebook lives in 3-custom-translator/notebook_publisher/.
# The tested publisher code lives in the sibling 3-custom-translator/streamlit_app/.
APP_DIR = pathlib.Path.cwd().parent / "streamlit_app"
if not (APP_DIR / "publisher.py").exists():
    # fall back to common locations if run from a different working dir
    for cand in [
        pathlib.Path.cwd() / "streamlit_app",
        pathlib.Path.cwd().parent / "streamlit_app",
        pathlib.Path.cwd(),
    ]:
        if (cand / "publisher.py").exists():
            APP_DIR = cand
            break
assert (APP_DIR / "publisher.py").exists(), f"publisher.py not found near {APP_DIR}"
sys.path.insert(0, str(APP_DIR))
os.chdir(APP_DIR)  # so get_config() picks up ./.env and relative manifest/translator paths

import publisher
from config import get_config
from auth import SCOPE_ARM, SCOPE_GRAPH, SCOPE_FOUNDRY

cfg = get_config()
print("publisher + config loaded from", APP_DIR)

## 2. The shared platform this agent will hook into

These are the **shared, one-time** resources. Every agent published (here or via Streamlit) reuses them — nothing per-agent here.

In [ ]:
print("=== Shared platform (reused by every agent) ===")
print("Subscription        :", cfg.subscription_id)
print("Resource group      :", cfg.resource_group)
print("APIM                :", cfg.apim_name, "/ API:", cfg.foundry_bot_api_name)
print("Bot-secrets KeyVault :", cfg.bot_secrets_keyvault_name)
print("Shared translator CA :", "your shared registry-driven router container")
print("Registry table       :", cfg.agent_registry_table_name, "@", cfg.translator_thread_table_url)
print()
print("=== Shared OAuth client (Entra app, pre-authorized on the MCP resource) ===")
print("SHARED_OAUTH_CLIENT_ID :", cfg.shared_oauth_client_id)
print("DEFAULT_MCP_SCOPE      :", cfg.default_mcp_scope)
print("OAUTH_REDIRECT_BASE_URL:", cfg.oauth_redirect_base_url)
print("Client secret set?     :", bool(cfg.shared_oauth_client_secret))

## 3. The Foundry agent to publish

Set the account / project / agent you want to wire to Teams. The project endpoint is derived from the account name.

In [ ]:
# ── EDIT THESE for your Foundry agent ────────────────────────────────
FOUNDRY_ACCOUNT = "<your-foundry-account>"   # AIServices account name
FOUNDRY_PROJECT = "<your-project>"           # Foundry project name
AGENT_ID        = "<your-agent-name>"        # the Foundry agent (Prompt Agent) name
DISPLAY_NAME    = "<Your Agent Display Name>"
BOT_SHORT_NAME  = "<your-agent-slug>"        # lowercase-hyphens; used for resource names
# ─────────────────────────────────────────────────────────────────────

FOUNDRY_PROJECT_ENDPOINT = f"https://{FOUNDRY_ACCOUNT}.services.ai.azure.com/api/projects/{FOUNDRY_PROJECT}"

err = publisher.validate_bot_short_name(BOT_SHORT_NAME)
assert err is None, err

inputs = publisher.PublishInputs(
    agent_id=AGENT_ID,
    display_name=DISPLAY_NAME,
    bot_short_name=BOT_SHORT_NAME,
    description_short=f"{DISPLAY_NAME} in Teams",
    description_full=f"{DISPLAY_NAME} published from Foundry to Teams via the shared translator container.",
    developer_name="<Your Org>",
    developer_website="https://www.example.com",
    developer_privacy="https://www.example.com/privacy",
    developer_terms="https://www.example.com/terms",
)
print("Will publish agent:", AGENT_ID, "->", FOUNDRY_PROJECT_ENDPOINT)

## 4. Token provider (local `az login` identity)

The publisher functions take a `(scope) -> token` callable. Locally we back it with `DefaultAzureCredential` pinned to your tenant (`AZURE_TENANT_ID` in `.env`).

In [ ]:
from azure.identity import DefaultAzureCredential

_tenant = os.getenv("AZURE_TENANT_ID") or cfg.entra_tenant_id
_cred = DefaultAzureCredential(interactive_browser_tenant_id=_tenant) if _tenant else DefaultAzureCredential()

def token_provider(scope: str) -> str:
    return _cred.get_token(scope).token

# sanity check
_ = token_provider(SCOPE_ARM)
print("token provider OK (ARM token acquired)")

## 5. Publish (shared route — NO per-agent container)

This runs the same `publisher.publish()` the Streamlit app calls:
1. create/reuse bot Entra app + fresh secret,
2. store secret in Key Vault,
3. merge the bot's AppId into APIM's `validate-jwt` audiences,
4. auto-derive auth mode from the agent's tools (`app` vs `oauth`),
5. create the Azure Bot + Teams channel (pointing at the **shared** APIM route),
6. **write the registry row** (what the shared container read-through loads),
7. build the Teams `.zip`.

> Requires VPN connected (Key Vault + Storage table are private-endpoint only).

In [ ]:
def progress(msg: str):
    print(msg)

result = publisher.publish(
    token_provider=token_provider,
    foundry_project_name=FOUNDRY_PROJECT,
    foundry_project_endpoint=FOUNDRY_PROJECT_ENDPOINT,
    inputs=inputs,
    progress=progress,
)

print("\n=== PUBLISH RESULT ===")
print("Bot name     :", result.bot_name)
print("Bot AppId    :", result.bot_app_id)
print("Secret in KV :", result.secret_kv_name, "/", result.secret_kv_secret_name)
print("Endpoint     :", result.endpoint)
print("Teams zip    :", result.teams_zip_name, f"({len(result.teams_zip_bytes)} bytes)")

## 6. Save the Teams app package

Upload this `.zip` in Teams (Apps → Manage your apps → Upload a custom app) to install the bot.

In [ ]:
out_dir = pathlib.Path.cwd() / "out"
out_dir.mkdir(exist_ok=True)
zip_path = out_dir / result.teams_zip_name
zip_path.write_bytes(result.teams_zip_bytes)
print("Saved Teams app package ->", zip_path)

## 7. What just happened (and what did NOT)

- ✅ A **registry row** was written for this bot → the **shared** container serves it (read-through, no restart).
- ✅ **No new Container App** was created — every agent runs on the one shared router container.
- ✅ The APIM audience list now accepts this bot's token; routing uses the shared templated operation.

### Verify no container was created
```powershell
az containerapp list -g <your-rg> --query "[?starts_with(name,'ca-bot')].name" -o table
```

### Verify the registry row
```powershell
az storage entity query --account-name <your-storage-account> --table-name agents --auth-mode login --query "items[].{bot:RowKey, agent:foundryAgentName, mode:authMode}" -o table
```

### Next
Upload the `.zip` to Teams and message the bot:
- **`app` mode + a passthrough MCP tool** (the proven default): Foundry returns a **consent card** ("Sign in required → Open consent") on first use; after **Allow access**, re-send and the agent answers **as you**.
- **`app` mode, no per-user tool**: the agent answers directly (container managed identity).
- **`oauth` mode** (explicit opt-in): the *container* shows its own **Sign in** card first, then forwards `x-ms-user-token`.